## Analysing SegFormer segmentation model
---

In this notebook, we are going to fine-tune SegFormerForSemanticSegmentation on a custom semantic segmentation dataset. In semantic segmentation, the goal for the model is to label each pixel of an image with one of a list of predefined classes.

## Imports 
---

In [ ]:
#external
from tqdm.notebook import tqdm

#model
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
import torch
from torch.utils.data import Dataset, DataLoader

#metrics
import evaluate

import torch.nn.functional as F

#utils
from src.utils.dataset import load_foodseg103, decode_image_from_bytes
from src.utils.visualization import predict_random_images

#constants
from src.constants.category_id import CATEGORY_ID

## Testing on custom dataset
---

In [ ]:
IMAGE_SIZE = 128
LEARNING_RATE = 0.0001
BATCH_SIZE = 8
NUM_EPOCHS = 10

### Defining Dataset

In [ ]:
class SemanticSegmentationFoodDataset(Dataset):
    def __init__(self, image_processor:SegformerImageProcessor, sample_size:int=None, data_type:str="train"):
        image_dataset = load_foodseg103(type=data_type, sample_size=sample_size)
        self.image_dataset = image_dataset
        self.image_processor = image_processor

    def __len__(self):
        return self.image_dataset.shape[0]
    
    def __getitem__(self, index):
        image_information = self.image_dataset.loc[index]
        image_decoded = decode_image_from_bytes(image_information["image"])
        mask = decode_image_from_bytes(image_information["label"])
        encoded_inputs = self.image_processor(image_decoded, mask, return_tensors="pt")
        for k,v in encoded_inputs.items():
          encoded_inputs[k].squeeze_() # remove batch dimension
        return encoded_inputs

In [ ]:
image_processor = SegformerImageProcessor(
    do_reduce_labels=False,
    size={"height": IMAGE_SIZE, "width": IMAGE_SIZE}
)

In [ ]:
train_dataset = SemanticSegmentationFoodDataset(image_processor, data_type="train")
validation_dataset = SemanticSegmentationFoodDataset(image_processor, data_type="validation")

In [ ]:
print(f"Number of training examples: {train_dataset.__len__()}")
print(f"Number of training examples: {validation_dataset.__len__()}")

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
validation_dataloader = DataLoader(validation_dataset, batch_size=BATCH_SIZE)

### Model definition

In [ ]:
num_classes = len(CATEGORY_ID)

In [ ]:
food_model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b0", #smaller model
    num_labels=num_classes,
    id2label=CATEGORY_ID,
    label2id={v: k for k, v in CATEGORY_ID.items()},
)

In [ ]:
for param in food_model.base_model.parameters():
    param.requires_grad = False

### Model training

In [ ]:
optimizer = torch.optim.AdamW(food_model.parameters(), lr=LEARNING_RATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
food_model.to(device);

metric = evaluate.load("mean_iou")

In [2]:
import shutil
import os

In [ ]:
food_model.train()
for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    for idx, batch in enumerate(tqdm(train_dataloader)):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)
        optimizer.zero_grad()
        outputs = food_model(pixel_values=pixel_values, labels=labels)
        loss, logits = outputs.loss, outputs.logits

        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

        #predict
        food_model.eval()
        with torch.no_grad():
            upsampled_logits = F.interpolate(logits, size=labels.shape[-2:], mode="bilinear", align_corners=False)
            predicted = upsampled_logits.argmax(dim=1)
            metric.add_batch(predictions=predicted.detach().cpu().numpy(), references=labels.detach().cpu().numpy())
        food_model.train()
        
        #evaluation
        if idx % 400 == 0:  
            metrics = metric._compute(
                    predictions=predicted.cpu(),
                    references=labels.cpu(),
                    num_labels=num_classes,
                    ignore_index=0,
                    reduce_labels=False
                )
            print(f"Epoch: {epoch+1} Batch: {idx}\nLoss: {round(loss.item(), 4)} | Mean_iou: {round(metrics['mean_iou'], 4)} | Mean accuracy: {round(metrics['mean_accuracy'], 4)}")
            metric = evaluate.load("mean_iou")
            food_model.save_pretrained("src/model")

### Testing in new image

In [ ]:
food_model.eval();

In [ ]:
predict_random_images(food_model, image_processor, validation_dataset, num_images=30)

## References
[1] https://github.com/NielsRogge/Transformers-Tutorials/tree/master/SegFormer